In [ ]:
from transformers import (
    BertForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    pipeline,
)
from datasets import Dataset, load_dataset



In [3]:
import pandas as pd

df = pd.read_csv("D:\\database\\csv files\\twitter_training.csv")
df2 = pd.read_csv("D:\\database\\csv files\\twitter_validation.csv")

In [58]:
model = BertForSequenceClassification.from_pretrained(
    "D:\\llms\\model\\llm\\sentimentbert-clf",
)
tokenizer = AutoTokenizer.from_pretrained("D:\\llms\\model\\llm\\sentimentbert-clf")

In [5]:
import re
import string


def preprocess(text):
    text = str(text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[@#]", "", text)
    return text.translate(str.maketrans("", "", string.punctuation))


df["tweets"] = df["tweets"].apply(preprocess)
df2["tweets"] = df2["tweets"].apply(preprocess)

In [6]:
train_dataset = Dataset.from_pandas(df)
test_dataset = Dataset.from_pandas(df2)

In [7]:
def tokenize_function(batch):
    tweets = []
    for t in batch["tweets"]:
        if t is None:
            tweets.append("")
        else:
            tweets.append(str(t))
    return tokenizer(
        tweets,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

In [8]:
texts = df["tweets"].tolist()
labels = df["labels"].tolist()

train_data = train_dataset.map(tokenize_function, batched=True)
train_data.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

test_data = test_dataset.map(tokenize_function, batched=True)
test_data.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/74681 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

In [9]:
trainarg = TrainingArguments(
    output_dir="D:\\AI\\DL\\outputs",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    learning_rate=2e-5,
    logging_steps=500,
    report_to=[],
)


trainer = Trainer(
    model=model,
    args=trainarg,
    train_dataset=train_data,
    # eval_dataset=test_data,
)

# trainer.train()

In [80]:
trainer.evaluate(eval_dataset=test_data)

d:\AI\tf-usage\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.1232370138168335,
 'eval_model_preparation_time': 0.0041,
 'eval_runtime': 17.9714,
 'eval_samples_per_second': 55.588,
 'eval_steps_per_second': 1.781}

In [15]:
import torch

In [61]:
text = "i am not so hapyy"

inputs = tokenizer(
    text, padding=True, truncation=True, max_length=128, return_tensors="pt"
)
probs = model(**inputs).logits[0]
import torch

probs = torch.softmax(probs, dim=0)
probs

tensor([0.0966, 0.5401, 0.2232, 0.1401], grad_fn=<SoftmaxBackward0>)

In [62]:
pip = pipeline("text-classification", model=model, tokenizer=tokenizer)
pip("i am not so hapyy")[0]

Device set to use cpu


{'label': 'LABEL_1', 'score': 0.5400802493095398}

In [84]:
# trainer.save_model(r"D:\llms\model\bert-clf")
# tokenizer.save_pretrained(r"D:\llms\model\brt-token")

In [94]:
from deep_translator import GoogleTranslator

translated = GoogleTranslator(source="gu", target="en").translate("મને બિલકુલ સમજાયું નથી.")
print(translated)

I don't understand at all.


In [101]:
from langdetect import detect


def detect_language(text: str) -> str:
    try:
        return detect(text)  # e.g. "fr", "hi", "es"
    except:
        return "en"


detect_language("મને બિલકુલ સમજાયું નથી.")

'gu'

In [8]:
from deep_translator import GoogleTranslator

translated = GoogleTranslator(source="auto", target="en").translate("Bonjour le monde")
print(translated)  # Output: Hello world

Hello world


In [7]:
from translate import Translator

translator = Translator(to_lang="en")
translation = translator.translate("Bonjour le monde")
print(translation)  # Output: Hello world

Hello World
